[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lanzlagman/intro-python-astro-PUP/blob/main/notebooks/01_setup_and_astropy_fundamentals.ipynb)

# Notebook 01: Basic Setup and Astropy Fundamentals

**Computational and Data-driven Astrophysics: A Practical Python Workshop**
PUP Physics Society · 15 August 2026

---

By the end of this notebook you will have **one function** that takes a star's temperature,
radius and distance and returns seven measurable properties.

| | Topic | Tool |
|---|---|---|
| 1 | Parallax and distance | plain Python |
| 2 | Why units break research code | `astropy.units` |
| 3 | Magnitudes and distance modulus | `numpy` |
| 4 | Blackbody luminosity | `numpy` |
| 5 | From one star to a catalogue | `astropy.table` |
| 6 | Where things are on the sky | `astropy.coordinates` |

No prior Python required. Cells marked **YOUR TURN** have one line for you to fill in.

## Part 0: Setup

Run this once. On Colab it installs what is missing; locally it usually does nothing.

In [1]:
# Colab already has numpy/matplotlib. Astropy is usually present too, but pin it anyway.
try:
    import astropy
except ImportError:
    !pip install -q astropy

import numpy as np
import astropy.units as u
from astropy.table import QTable
from astropy.coordinates import SkyCoord
import astropy.constants as const

print("numpy  ", np.__version__)
print("astropy", astropy.__version__)

numpy   2.2.6
astropy 8.0.1


---
## Part 1: Parallax

As Earth orbits the Sun, a nearby star shifts against the distant background. Half of that
maximum angular shift is the **parallax angle** $p$. The parsec is defined to make the
relation as simple as possible:

$$ d = \frac{1}{p''}\ \text{pc} $$

This only works if $p$ is in **arcseconds** and $d$ comes out in **parsecs**. That
restriction bites in Part 2.

In [2]:
def distance_from_parallax(p_arcsec):
    """Distance in parsecs from a parallax angle in arcseconds."""
    return 1.0 / p_arcsec


# Check against Bessel's 1838 measurement of 61 Cygni
d_61cyg = distance_from_parallax(0.316)
print(f"61 Cygni : p = 0.316 arcsec  ->  d = {d_61cyg:.2f} pc")
print("Published value: 3.16 pc.  Match:", np.isclose(d_61cyg, 3.16, atol=0.01))

# The parallax angle for Sirius is 0.379 arcsec
d_sirius = distance_from_parallax(0.379)
print(f"\nSirius   : p = 0.379 arcsec  ->  d = {d_sirius:.3f} pc")
print(f"                              = {d_sirius * 3.2615638:.2f} ly")

61 Cygni : p = 0.316 arcsec  ->  d = 3.16 pc
Published value: 3.16 pc.  Match: True

Sirius   : p = 0.379 arcsec  ->  d = 2.639 pc
                              = 8.61 ly


---
## Part 2: The units trap

Gaia reports parallax in **milliarcseconds**, not arcseconds. Paste a Gaia number straight
into the formula above and nothing crashes; the answer is just wrong by a factor of 1000.

Run the next cell and compare the two numbers.

In [3]:
# Gaia reports the parallax of Sirius as about 379 mas.
p_gaia = 379.0          # milliarcseconds  <-- note the unit

wrong = distance_from_parallax(p_gaia)
print(f"Naive  : {wrong:.6f} pc   <- Sirius is NOT 2.6 milliparsecs away")
print(f"Correct: {distance_from_parallax(p_gaia / 1000):.3f} pc")

Naive  : 0.002639 pc   <- Sirius is NOT 2.6 milliparsecs away
Correct: 2.639 pc


### The fix: make the units part of the number

`astropy.units` attaches a physical unit to a value. The result is a `Quantity`, which
carries its unit through every operation and **refuses** conversions that make no sense.

In [4]:
p = 379.0 * u.mas               # a Quantity: value + unit
print("p =", p)

# .to() converts. Astropy knows mas -> arcsec.
print("p =", p.to(u.arcsec))

# Astropy even knows the parallax <-> distance relationship directly:
d = p.to(u.pc, equivalencies=u.parallax())
print("d =", d.round(3))
print("d =", d.to(u.lightyear).round(2))

# And it stops you from doing something meaningless:
try:
    bad = p.to(u.kg)
except u.UnitConversionError as err:
    print("\nBlocked, as it should be:")
    print(" ", err)

p = 379.0 mas
p = 0.379 arcsec
d = 2.639 pc
d = 8.61 lyr

Blocked, as it should be:
  'mas' (angle) and 'kg' (mass) are not convertible


> **Takeaway.** Plain floats let a factor-of-1000 error through silently. `Quantity` objects
> turn it into an exception on the line where the mistake happened.

---
## Part 3: Magnitudes and distance modulus

The magnitude scale runs backwards (brighter = smaller number) and is defined so that
**five magnitudes = a factor of 100 in flux**:

$$ \frac{F_2}{F_1} = 100^{(m_1 - m_2)/5} $$

**Absolute magnitude** $M$ is the apparent magnitude a star *would* have at 10 pc. Combining
that with the inverse square law $F = L/4\pi r^2$ gives the **distance modulus**:

$$ m - M = 5\log_{10}(d) - 5, \qquad d \text{ in pc} $$

In [5]:
def distance_modulus(d_pc):
    """Distance modulus:  m - M = 5 log10(d) - 5"""
    return 5 * np.log10(d_pc) - 5

def absolute_magnitude(m_apparent, d_pc):
    """Rearranged distance modulus."""
    return m_apparent - distance_modulus(d_pc)


# The Sun, as a worked check
m_sun, d_sun_pc = -26.83, 4.848e-6
M_sun = absolute_magnitude(m_sun, d_sun_pc)

print(f"Sun: M = {M_sun:+.2f}          (accepted value: +4.74)")
print(f"Sun: m - M = {distance_modulus(d_sun_pc):.2f}   (accepted value: -31.57)")

Sun: M = +4.74          (accepted value: +4.74)
Sun: m - M = -31.57   (accepted value: -31.57)


### ✏️ YOUR TURN

Find the distance modulus for Sirius. You already have `d_sirius` from Part 1.
Replace `None` on the marked line.

In [6]:
# YOUR TURN: call distance_modulus() on the distance to Sirius
dm_sirius = None          # <-- replace None

# ---- check ----
if dm_sirius is None:
    print("Fill in the line above, then re-run.")
else:
    print(f"Sirius distance modulus m - M = {dm_sirius:.3f}")
    m_sirius = -1.46
    print(f"Sirius absolute magnitude M   = {m_sirius - dm_sirius:+.2f}")
    print("Expected about -2.89 for m - M, and M = +1.43.")
    print("Sirius is CLOSER than 10 pc, so its distance modulus is negative.")

Fill in the line above, then re-run.


---
## Part 4: One star, seven numbers

A star radiates approximately as a blackbody at its effective temperature $T_e$. Two results
do most of the work:

$$\lambda_{max} T = 0.002897755\ \text{m K} \qquad \text{(Wien)}$$

$$L = 4\pi R^2 \sigma T_e^4 \qquad \text{(Stefan–Boltzmann)}$$

Together with $M = M_\odot - 2.5\log_{10}(L/L_\odot)$, these take us from *temperature,
radius, distance* to almost everything an observer can measure.

> ### Problem 9, p.94: Dschubba (δ Sco)
> 
> Consider a model of the star Dschubba ($\delta$ Sco), the center star in the head of the constellation Scorpius. Assume that Dschubba is a spherical blackbody with a surface temperature of $28,000\text{ K}$ and a radius of $5.16 \times 10^9\text{ m}$. Let this model star be located at a distance of $123\text{ pc}$ from Earth. Determine the following for the star:
> 
> **(a)** Luminosity.  
> **(b)** Absolute bolometric magnitude.  
> **(c)** Apparent bolometric magnitude.  
> **(d)** Distance modulus.  
> **(e)** Radiant flux at the star's surface.  
> **(f)** Radiant flux at Earth's surface (compare this with the solar irradiance).  
> **(g)** Peak wavelength $\lambda_{\text{max}}$.

In [7]:
# Constants, straight from astropy (no hand-typed values to mistype)
SIGMA = const.sigma_sb          # Stefan-Boltzmann constant
L_SUN = 3.839e26 * u.W          # solar luminosity, textbook value
M_BOL_SUN = 4.74                # solar bolometric magnitude
WIEN = 0.002897755 * u.m * u.K  # Wien's displacement constant


def star_properties(
    T_eff, 
    R, 
    d
):
    """All seven observable properties, in one pass.

    Parameters are Quantities, so units are checked for us.
    """
    L = 4 * np.pi * R**2 * SIGMA * T_eff**4              # Stefan-Boltzmann
    M_bol = M_BOL_SUN - 2.5 * np.log10(L / L_SUN)        # bolometric magnitude
    dist_mod = 5 * np.log10(d / (10 * u.pc))             # distance modulus
    m_bol = M_bol + dist_mod
    F_surface = L / (4 * np.pi * R**2)                   # inverse square law at r = R
    F_earth = L / (4 * np.pi * d**2)                     # inverse square law at r = d
    lambda_max = WIEN / T_eff                            # Wien

    return {
        "(a) luminosity":         L.to(u.W),
        "    L / L_sun":          (L / L_SUN).decompose(),
        "(b) M_bol":              M_bol,
        "(c) m_bol":              m_bol,
        "(d) distance modulus":   dist_mod,
        "(e) flux at surface":    F_surface.to(u.W / u.m**2),
        "(f) flux at Earth":      F_earth.to(u.W / u.m**2),
        "(g) lambda_max":         lambda_max.to(u.nm),
    }

In [8]:
dschubba = star_properties(
    T_eff=28000 * u.K,
    R=5.16e9 * u.m,
    d=123 * u.pc,
)

print("Dschubba (delta Scorpii)\n" + "-" * 24)
for key, value in dschubba.items():
    print(f"{key:24s} {value:.4g}")

print("\nSanity checks:")
print(f"  ~{float(dschubba['    L / L_sun']):.0f}x the Sun's luminosity - plausible for a hot B star")
print(f"  lambda_max = {dschubba['(g) lambda_max']:.0f} - ultraviolet, so we see only its blue tail")

Dschubba (delta Scorpii)
------------------------
(a) luminosity           1.166e+31 W
    L / L_sun            3.038e+04
(b) M_bol                -6.466
(c) m_bol                -1.017
(d) distance modulus     5.45
(e) flux at surface      3.485e+10 W / m2
(f) flux at Earth        6.442e-08 W / m2
(g) lambda_max           103.5 nm

Sanity checks:
  ~30376x the Sun's luminosity - plausible for a hot B star
  lambda_max = 103 nm - ultraviolet, so we see only its blue tail


> **Note the payoff.** Solar irradiance at Earth is about 1365 W m⁻². Compare it with the
> flux we just computed from Dschubba: a far more luminous star delivering a vanishingly
> small flux here. That ratio *is* the inverse square law, and it is why astronomy is hard.

---
## Part 5: From one star to a catalogue

Measuring a full spectrum per star is expensive. Measuring brightness through two filters is
cheap, and the difference between them, the **colour index**, is a temperature proxy.
Notebook 02 pulls thousands of these; here we handle five.

An astropy table is a spreadsheet where **each column carries its own unit**, so everything
we just did for one star happens for all rows at once, with no loop.

> ⚠️ Use **`QTable`**, not `Table`. In a `QTable` each column *is* a `Quantity`, so units
> survive arithmetic. In a plain `Table` they do not, and `R**2` silently loses the squaring.

In [9]:
stars = QTable({
    "name":   ["Sirius A", "Vega", "Rigel", "Betelgeuse", "Proxima Cen"],
    "T_eff":  [9940, 9602, 12100, 3600, 3042] * u.K,
    "R":      [1.71, 2.36, 78.9, 887.0, 0.154] * const.R_sun,
    "plx":    [379.21, 130.23, 3.78, 5.95, 768.50] * u.mas,
})

# One vectorised call replaces a five-iteration loop.
stars["d"] = stars["plx"].to(u.pc, equivalencies=u.parallax())

stars["L"] = (4 * np.pi * stars["R"]**2 * SIGMA * stars["T_eff"]**4).to(u.W)
stars["M_bol"] = M_BOL_SUN - 2.5 * np.log10(stars["L"] / L_SUN)
stars["m_bol"] = stars["M_bol"] + 5 * np.log10(stars["d"] / (10 * u.pc))

for col in ("d", "L", "M_bol", "m_bol"):
    stars[col].info.format = ".3g"

stars["name", "T_eff", "d", "L", "M_bol", "m_bol"].pprint()
print("\nColumns keep their units:", stars["d"].unit, stars["L"].unit)
print("Compare M_bol against the Sun's +4.74.")

    name     T_eff   d      L     M_bol m_bol 
               K     pc     W                 
----------- ------- ---- -------- ----- ------
   Sirius A  9940.0 2.64 9.84e+27  1.22  -1.68
       Vega  9602.0 7.68 1.63e+28 0.668 0.0946
      Rigel 12100.0  265  4.6e+31 -7.96 -0.844
 Betelgeuse  3600.0  168 4.56e+31 -7.95  -1.82
Proxima Cen  3042.0  1.3    7e+23  11.6   7.16

Columns keep their units: pc W
Compare M_bol against the Sun's +4.74.


### ✏️ YOUR TURN

Add a `lambda_max` column using Wien's law. The constant is already defined as `WIEN`, and
`stars["T_eff"]` is a whole column; you do **not** need a loop.

In [10]:
# YOUR TURN: one line, no loop.  Hint: WIEN / stars["T_eff"], then .to(u.nm)
lambda_max = None      # <-- replace None

# ---- check ----
if lambda_max is None:
    print("Fill in the line above, then re-run.")
else:
    stars["lambda_max"] = lambda_max
    stars["lambda_max"].info.format = ".0f"
    stars["name", "T_eff", "lambda_max"].pprint()
    print("\nRigel peaks in the ultraviolet, Proxima deep in the infrared.")
    print("For reference the Sun (5777 K) peaks near 500 nm - the middle of")
    print("human vision. That is not a coincidence.")

Fill in the line above, then re-run.


---
## Part 6: Where is it on the sky?

The **equatorial coordinate system** projects Earth's latitude and longitude onto the sky
without rotating with the planet. **Declination** $\delta$ plays the role of latitude;
**right ascension** $\alpha$ plays the role of longitude, measured in hours, minutes and
seconds.

Every archive query you will ever write starts with a position in this system. `SkyCoord` is
the object that holds one.

In [11]:
# The Pleiades (M45) - our target for the rest of the workshop.
# A nearby open cluster in Taurus, about 130 pc away.
pleiades = SkyCoord(ra="03h47m00s", dec="+24d07m00s", frame="icrs")

print("Pleiades (M45)")
print("  sexagesimal :", pleiades.to_string("hmsdms"))
print("  degrees     :", pleiades.to_string("decimal"))
print(f"  ra  = {pleiades.ra.deg:.4f} deg")
print(f"  dec = {pleiades.dec.deg:.4f} deg")

# Same point, different reference frame - one attribute access
print("\n  galactic    :", pleiades.galactic.to_string("decimal"))

# Angular separation between two objects: one method call, no spherical trig by hand
sirius = SkyCoord.from_name("Sirius") if False else SkyCoord(
    ra="06h45m08.9s", dec="-16d42m58s", frame="icrs")
print(f"\nPleiades to Sirius: {pleiades.separation(sirius).deg:.2f} degrees apart")

Pleiades (M45)
  sexagesimal : 03h47m00s +24d07m00s
  degrees     : 56.75 24.1167
  ra  = 56.7500 deg
  dec = 24.1167 deg

  galactic    : 166.571 -23.5212

Pleiades to Sirius: 59.63 degrees apart


> **Why `SkyCoord` and not two floats?** Because `separation()` is doing spherical
> trigonometry that is easy to get wrong by hand, and because the frame conversion is the
> difference between "24 degrees" meaning declination and meaning galactic latitude.

---
## What you built, and where it goes

| You wrote | Next used in |
|---|---|
| `astropy.units` discipline | every notebook |
| Distance modulus | **NB04:** apparent into absolute magnitude |
| Stefan–Boltzmann | **NB04:** reading a colour–magnitude diagram |
| `Table` with units | **NB02:** what every archive query returns |
| `SkyCoord` | **NB02:** the centre of our cone search |

**Notebook 02** takes the `SkyCoord` we just made, hands it to the European Space Agency, and
gets back a few thousand stars.

---
### Sources

The physics here follows Carroll & Ostlie, *An Introduction to Modern Astrophysics*, 2nd ed.: §1.3 (Positions on the Celestial Sphere), §3.1 (Stellar Parallax), §3.2 (The Magnitude Scale), §3.4 (Blackbody Radiation), §3.6 (The Color Index). Worked values and exercises are adapted from those sections and their problem set.

### If you want to go deeper
- Pasha & Agostino, *Python for Astronomers*: Python Bootcamp Day 1 & 2 · <https://prappleizer.github.io/>
- Learn Astropy: Units & Quantities, and Coordinates guides · <https://learn.astropy.org/>